In [ ]:
# =============================================================================
# Automated ATAS Processing Pipeline (CWS / CWNS)
#
# PURPOSE:
# This notebook processes audio files to automatically detect speech and pause
# events using signal processing and segmentation techniques.
#
# INPUTS:
# - Folder of .wav audio files
# - CSV file listing filenames to process
#
# OUTPUTS:
# - *_50_150.csv → event-level segmentation files
# - Group-level statistics CSV (speech/pause metrics per file)
#
# PROCESSING STEPS:
# 1. Match audio files with CSV entries
# 2. Apply bandpass filtering
# 3. Perform noise reduction
# 4. Normalize audio amplitude
# 5. Segment audio into time windows
# 6. Detect speech and pause events
# 7. Compute event-level and file-level statistics
#
# NOTE:
# The *_50_150.csv files generated here are used as input to the
# ATAS_event_relabelling notebook for correction.
# =============================================================================

In [ ]:
import os, glob, wave, itertools, statistics
import numpy as np, pandas as pd, librosa, librosa.display, matplotlib.pyplot as plt
import noisereduce as nr, IPython.display as ipd, matplotlib.patches as mpatches
import soundfile as sf
from scipy.io import wavfile
from scipy.signal import butter, filtfilt
from scipy.stats import variation
from pydub import AudioSegment, silence

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

In [ ]:
%run -i "$base_dir/Scripts/ATAS_functions_CWS_CWNS.ipynb" # Change to required destination    

In [ ]:
pause_time_threshold = 50
speech_time_threshold = 100
segment_size_t = 1 # smaller segments to calculate mean within longer segment_size_t_windows
segment_size_t_window = 3 # segment size in seconds

In [ ]:
def run_pipeline(Audio_files_clipped, csv_name):

    # -------------------------------------------------------------------------
    # Create output folder for individual event-level CSV files
    # -------------------------------------------------------------------------
    individual_output_dir = os.path.join(base_dir, 'Individual_OutputCSV_Files')
    os.makedirs(individual_output_dir, exist_ok=True)

    # -------------------------------------------------------------------------
    # Read metadata CSV containing the filenames to be processed
    # -------------------------------------------------------------------------
    df_times = pd.read_csv(csv_name)

    # Print column names to confirm the expected filename column is present
    print("CSV columns:", df_times.columns.tolist())

    # Extract filenames listed in the CSV
    # Only these files should be included in processing
    csv_files = df_times['File_name'].astype(str).tolist()

    # -------------------------------------------------------------------------
    # Get all .wav files that are actually present in the audio folder
    # -------------------------------------------------------------------------
    os.chdir(Audio_files_clipped)
    all_files = glob.glob("*.wav")

    # -------------------------------------------------------------------------
    # Match CSV filenames with actual audio files in the folder
    # This prevents processing extra .wav files that are not listed in the CSV
    # -------------------------------------------------------------------------
    matched_files = [
        file for file in all_files
        if file in csv_files
    ]

    # Print matching summary for verification
    print("Files in audio folder:", len(all_files))
    print("Files listed in CSV:", len(csv_files))
    print("Matched files to process:", matched_files)

    # Stop if no files match between the CSV and audio folder
    if len(matched_files) == 0:
        print("No matching files found. Check filename column and exact filenames.")
        return

    # Store summary/statistics output for all processed files
    df_together = []

    # -------------------------------------------------------------------------
    # Process each matched audio file
    # -------------------------------------------------------------------------
    for f_name_current in matched_files:
        print(f"Processing file: {f_name_current}")

        # Set analysis window to include the full duration of the clipped audio
        start_time = 0.000
        end_time = librosa.get_duration(path=f_name_current)
        file_name_org = f_name_current

        # ---------------------------------------------------------------------
        # Apply bandpass filter to retain the desired frequency range
        # ---------------------------------------------------------------------
        infile = os.path.join(Audio_files_clipped, file_name_org)
        name, ext = os.path.splitext(file_name_org)
        outfile_name_proc = f"{name}_proc{ext}"
        outfile = os.path.join(Audio_files_clipped, outfile_name_proc)

        process_file(
            infile=infile,
            outfile=outfile,
            lowcut=80.0,
            highcut=11000.0
        )

        # Load filtered audio and apply noise reduction
        x, sr = load_display_file_wf(outfile_name_proc)
        x_nr, sr_nr, wav_file_nr = reduce_file_noise_wf(outfile_name_proc, x, sr)

        # ---------------------------------------------------------------------
        # Normalize noise-reduced audio to a consistent peak amplitude
        # ---------------------------------------------------------------------
        file_name_org = wav_file_nr
        infile = os.path.join(Audio_files_clipped, file_name_org)
        name, ext = os.path.splitext(file_name_org)
        outfile_name = f"{name}_norm{ext}"
        outfile = os.path.join(Audio_files_clipped, outfile_name)

        normalize_audio_file(infile, outfile, peak_db=-1.0)

        # ---------------------------------------------------------------------
        # Trim/load normalized audio for event detection
        # ---------------------------------------------------------------------
        x_nr_ex, sr_nr_ex, trim_file, f_trim_file, trim_file_viz, f_name_2 = wav_file_trim_wf(
            outfile_name, start_time, end_time
        )

        # ---------------------------------------------------------------------
        # Detect speech and pause/silence segments using window-based detection
        # ---------------------------------------------------------------------
        all_seg_details_silence_p_file, all_seg_details_silence_rem_file, seg_time_leng = (
            event_detection_seg_wf_window(
                x_nr_ex, sr_nr_ex, segment_size_t_window, trim_file
            )
        )

        # ---------------------------------------------------------------------
        # Classify detected segments into proper speech/pause events
        # using the defined speech and pause duration thresholds
        # ---------------------------------------------------------------------
        silence_p_final_end, silence_rem_final_end, ddf, long_p, short_p, long_p_durations, short_p_durations = (
            detect_proper_events(
                all_seg_details_silence_p_file,
                all_seg_details_silence_rem_file,
                speech_time_threshold,
                pause_time_threshold,
                f_name_current,
                seg_time_leng
            )
        )

        # ---------------------------------------------------------------------
        # Calculate event-level and file-level speech/pause statistics
        # ---------------------------------------------------------------------
        df, ddf = event_statistics_wf(
            f_name_current,
            f_trim_file,
            x_nr_ex,
            sr_nr_ex,
            silence_p_final_end,
            silence_rem_final_end,
            speech_time_threshold,
            pause_time_threshold,
            ddf,
            long_p,
            short_p
        )

        # ---------------------------------------------------------------------
        # Save individual event-level CSV for this file
        # This file is later used for correction / non-speech event review
        # ---------------------------------------------------------------------
        f_name_1 = f_name_current.rsplit('.', maxsplit=1)[0]
        ddf_csv_file = os.path.join(
            individual_output_dir,
            f_name_1 + '_50_150.csv'
        )
        ddf.to_csv(ddf_csv_file, index=False)

        # Append file-level summary statistics for combined group output
        df_list = df.values.tolist()
        df_together.append(df_list[0])

    # -------------------------------------------------------------------------
    # Combine summary statistics from all processed files into one DataFrame
    # -------------------------------------------------------------------------
    df_together_csv = pd.DataFrame(
        df_together,
        columns=[
            'File_Name',
            'Speech Time Threshold_ms',
            'Pause Time Threshold_ms',
            'Percent_Pause',
            'Percent_Speech',
            'Total_Duration_Unclipped_s',
            'Total_Duration_Clipped_s',
            'Speech_Duration_s',
            'Pause_Duration_s',
            'Speech_Events',
            'Pause_Events',
            'Mean Speech_s',
            'Std Dev Speech',
            'CV Speech',
            'Mean Pause_s',
            'Std Dev Pause',
            'CV Pause'
        ]
    )

    # -------------------------------------------------------------------------
    # Save combined file-level statistics output for the group
    # -------------------------------------------------------------------------
    stat_dir = os.path.join(base_dir, 'Stat_csv_files')
    os.makedirs(stat_dir, exist_ok=True)

    group_name = os.path.basename(csv_name).split('_')[0]
    file_tog_name = f'{group_name}_All_files_together_50_ms_1_win_50_150_csv.csv'
    csv_file = os.path.join(stat_dir, file_tog_name)

    df_together_csv.to_csv(csv_file, index=False, header=True)

    print(f"Saved combined output to: {csv_file}")

In [ ]:
run_pipeline(Audio_files_clipped = base_dir + '/Audio_files_clipped',
    csv_name = base_dir + '/Stat_csv_files/CWNS_final.csv'
)

In [ ]:
run_pipeline(Audio_files_clipped = base_dir + '/Audio_files_clipped',
    csv_name = base_dir + '/Stat_csv_files/CWS_final.csv'
)